# 2.11 Matrix free operator application
Usually, we assemble matrices in sparse matrix format. Certain methods allow us to improve performance and to reduce memory requirements considerably by avoiding building and storing the system matrix. The downside of this approach is that it is more difficult to build preconditioners.

## Hybrid mixed method

Consider the hybrid mixed method

$$
\begin{array}{ccccl}
A \sigma & + & B^T u & = & f \\
B \sigma & & & = & g
\end{array}
$$

where the matrix $A$ comes from

$$
\int \sigma \tau dx
$$

and the matrix $B$ from

$$
\sum_T \int_T \operatorname{div} \, \sigma \, v_T - \int_{\partial T} \sigma_n v_F
$$

For $\sigma$ we use a `VectorL2` space, with a Piola mapping to the physical elements

In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
from ngsolve.fem import MixedFE

mesh = Mesh(unit_square.GenerateMesh(maxh=0.3))
order=1
Sigma = VectorL2(mesh, order=order, piola=True)
Vt = L2(mesh, order=order-1)
Vf = FacetFESpace(mesh, order=order)
V = Vt*Vf
sigma,tau = Sigma.TnT()
(ut,uf), (vt,vf) = V.TnT()

In [ ]:
a = BilinearForm(sigma*tau * dx)

n = specialcf.normal(mesh.dim)
dS = dx(element_boundary=True)
b = BilinearForm(div(sigma) * vt * dx - sigma*n*vf * dS).Assemble();

Compute the element matrices ...

We should observe that the element matrices of the $A$ operator are block-diagonal with $2\times 2$ blocks, and that each element matrix of $B$ belongs to one of two equivalence classes.

In [ ]:
for el in mesh.Elements():
    felS = Sigma.GetFE(el)
    trafo = mesh.GetTrafo(el)
    elmat = a.integrators[0].CalcElementMatrix(felS, trafo)
    print (elmat)

In [ ]:
for el in mesh.Elements():
    felS = Sigma.GetFE(el)
    felV = V.GetFE(el)
    fel = MixedFE(felS, felV)
    trafo = mesh.GetTrafo(el)

    # try integrators 0 and 1 ...
    elmat = b.integrators[0].CalcElementMatrix(fel, trafo)
    print (elmat)

## Geometry-free matrix multiplication
NGSolve can benefit from the geometry-independent element matrices. Just one matrix per equivalence class is stored, and the matrix-vector product is performed for all elements simultaneously using dense matrix algebra.

In [ ]:
from netgen.csg import *
geom = CSGeometry()
geom.Add (Sphere(Pnt(50,50,50),80) \
          -Cylinder(Pnt(-100,0,0),Pnt(200,0,0), 40) \
          -Cylinder(Pnt(100,-100,100),Pnt(100,200,100),40)
          -Cylinder(Pnt(0,100,-100), Pnt(0,100,200),40)
          -Sphere(Pnt(50,50,50),50))
# geom.Draw()

mesh = Mesh(geom.GenerateMesh(maxh=25)).Curve(5)
# Draw (mesh)

The same as above, with the `geom_free` flag for the `BilinearForm`:

In [ ]:
order=6
Sigma = VectorL2(mesh, order=order, piola=True, tp=True)
Vt = L2(mesh, order=order-1)
Vf = FacetFESpace(mesh, order=order, dirichlet=[2])
V = Vt*Vf
print ("Sigma.ndof =", Sigma.ndof, ", V.ndof =", V.ndof)
sigma,tau = Sigma.TnT()
(ut,uf), (vt,vf) = V.TnT()

n = specialcf.normal(mesh.dim)
dS = dx(element_boundary=True)
b = BilinearForm(div(sigma) * vt * dx - sigma*n*vf * dS, geom_free=True).Assemble();

Check the timing with `geom_free=True/False`:

In [ ]:
vx = b.mat.CreateRowVector()
vy = b.mat.CreateColVector()
vx[:] = 1

%timeit vy.data = b.mat * vx

`L2` finite element spaces have a `Mass` method generating a linear operator for the multiplication with the block-diagonal mass matrix. In the case of variable coefficients or curved elements it uses numerical integration. It also provides an inverse operator:

In [ ]:
ainv = Sigma.Mass(rho=1).Inverse()

Check the timing with `tp=True/False`:

In [ ]:
vx = ainv.CreateRowVector()
vy = ainv.CreateColVector()
vx[:] = 1

%timeit vy.data = ainv * vx

Combine the operators to form the Schur complement, which is actually a discretization of the Laplace operator:

In [ ]:
Laplace = b.mat @ ainv @ b.mat.T

In [ ]:
f = LinearForm(1*vt*dx).Assemble()

proj = Projector(V.FreeDofs(), True)

gfu = GridFunction (V)
from time import time
start = time()
with TaskManager():
    inv = solvers.CGSolver(Laplace, proj, maxiter=5000)
    gfu.vec.data = inv * f.vec
print ("iterations =", inv.iterations, ", time =", time()-start)

The iteration count is very high since we used only the most trivial preconditioner. We can add a low order coarse-grid preconditioner to get rid of the mesh dependency, and also local inverses to improve the $p$-dependency.

In [ ]:
Draw (gfu.components[1], mesh, "gfu1", draw_vol=False);